# Late Fusion Simulation — Facial + Q-CHAT ASD Detection

**Multimodal fusion feasibility analysis** combining two independently trained models:

| Stream | Source | Model | N |
|--------|--------|-------|---|
| Facial | `sector1/Stage_4` | VGG-Face CNN → 256-D embedding → classifier | 3,917 |
| Q-CHAT-10 | `sector2/Stage_2` | XGBoost on parent-only questionnaire | 1,838 |

**Fusion rule:** `P_final = α × P_facial + (1−α) × P_qchat`

**Sections:**
- **A** — Load data, LogReg embedding probe, standalone baselines
- **B** — Alpha sweep (21 values × 500 MC iterations × 300 stratified pairs)
- **C** — Failure mode complementarity analysis
- **D** — Threshold sweep at optimal alpha
- **E** — 5 plots saved to `sector3/plots/`
- **F** — Clinical interpretation
- **G** — Save `fusion_simulation_summary.json`

---
## ⚠️ METHODOLOGICAL NOTE — Mandatory Disclaimer

> **The facial dataset (`sector1/Stage_4`) and the Q-CHAT-10 dataset (`sector2/Stage_2`) contain NO patient overlap — they are entirely independent populations collected separately.**
>
> The fusion analysis below is a **simulation** that estimates how a combined system might behave at a population level. For each Monte Carlo iteration, ASD-positive cases are independently sampled from the facial pool and the Q-CHAT pool and paired by class label — not by individual identity. Non-ASD cases are paired the same way. The fused probability `P_final = α × P_facial + (1−α) × P_qchat` is computed on these synthetic pairs and evaluated against the shared class label.
>
> Results describe **expected performance distributions** under a hypothetical dual-modal deployment, **not** a validation on matched individuals. This is standard practice in multimodal fusion feasibility research when prospectively paired ground-truth data is unavailable. All conclusions must be interpreted as probabilistic estimates of system-level behaviour only.
>
> A prospective study with the same children assessed by both modalities would be required for definitive paired validation.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, fbeta_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

# ─── Constants ────────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_MC          = 500       # Monte Carlo iterations
N_PAIRS       = 300       # pairs per iteration
N_PAIRS_ASD   = N_PAIRS // 2   # 150 ASD pairs
N_PAIRS_NON   = N_PAIRS - N_PAIRS_ASD  # 150 non-ASD pairs
ALPHA_VALUES  = np.arange(0.0, 1.05, 0.05)  # 21 alpha values

# ─── Paths ────────────────────────────────────────────────────────────────────
# Assumes notebook is run from its own directory (sector3/)
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent  # sector3→autisumDetect→mlModels→project

SECTOR1_DATA = (PROJECT_ROOT / 'mlModels' / 'autisumDetect' /
                'sector1' / 'Stage_4' / 'data' / 'ImgFeatures_Stage4.csv')
SECTOR2_DATA = (PROJECT_ROOT / 'mlModels' / 'autisumDetect' /
                'sector2' / 'Stage_2' / 'data' / 'qchat_probabilities_stage2.csv')
SECTOR3_DIR  = PROJECT_ROOT / 'mlModels' / 'autisumDetect' / 'sector3'
PLOTS_DIR    = SECTOR3_DIR / 'plots'
RESULTS_DIR  = SECTOR3_DIR / 'results'

for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Facial data  : exists={SECTOR1_DATA.exists()}')
print(f'Q-CHAT data  : exists={SECTOR2_DATA.exists()}')
print(f'Plots dir    : {PLOTS_DIR}')
print(f'Results dir  : {RESULTS_DIR}')
print(f'MC config    : {N_MC} iterations x {N_PAIRS} pairs x {len(ALPHA_VALUES)} alpha values')

---
## Section A — Load Data & Standalone Baselines

In [ ]:
# ─── Load facial stream ───────────────────────────────────────────────────────
df_img = pd.read_csv(SECTOR1_DATA)
FEAT_COLS = [f'V_{i}' for i in range(256)]

print('Facial dataset (sector1/Stage_4):')
print(f'  Total rows    : {len(df_img)}')
print(f'  ASD (label=1) : {(df_img["Actual_Label"]==1).sum()}')
print(f'  Non-ASD (0)   : {(df_img["Actual_Label"]==0).sum()}')
print(f'  ASD prevalence: {(df_img["Actual_Label"]==1).mean():.1%}')

# ─── Load Q-CHAT stream ──────────────────────────────────────────────────────
df_qc = pd.read_csv(SECTOR2_DATA)

print('\nQ-CHAT dataset (sector2/Stage_2, parent-only):')
print(f'  Total rows    : {len(df_qc)}')
print(f'  ASD (label=1) : {(df_qc["true_label"]==1).sum()}')
print(f'  Non-ASD (0)   : {(df_qc["true_label"]==0).sum()}')
print(f'  ASD prevalence: {(df_qc["true_label"]==1).mean():.1%}')

print('\nNote: These are independent populations — no patient overlap.')

In [ ]:
# ─── LogReg probe on 256-D VGG-Face embeddings (out-of-fold — no leakage) ────
X_img = df_img[FEAT_COLS].values
y_img = df_img['Actual_Label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_img)

logreg = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
cv5    = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Running 5-fold cross_val_predict on LogReg probe (256-D embeddings)...')
logreg_oof_probs = cross_val_predict(
    logreg, X_scaled, y_img, cv=cv5, method='predict_proba'
)[:, 1]

logreg_auc = roc_auc_score(y_img, logreg_oof_probs)
print(f'LogReg probe OOF AUC : {logreg_auc:.4f}')
print('(OOF = out-of-fold cross-validation, no train/test leakage)')

In [ ]:
# ─── Standalone baseline metrics ─────────────────────────────────────────────
facial_auc = roc_auc_score(df_img['Actual_Label'], df_img['Model_Prob'])
qchat_auc  = roc_auc_score(df_qc['true_label'],   df_qc['P_ASD'])

def metrics_at_thresh(y_true, y_prob, thresh=0.5):
    y_pred = (np.array(y_prob) >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall      = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f2          = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    return dict(recall=recall, precision=precision,
                specificity=specificity, f2=f2)

facial_m  = metrics_at_thresh(df_img['Actual_Label'], df_img['Model_Prob'])
qchat_m   = metrics_at_thresh(df_qc['true_label'],   df_qc['P_ASD'])
logreg_m  = metrics_at_thresh(y_img, logreg_oof_probs)

baseline_df = pd.DataFrame({
    'Stream'     : ['VGG-Face CNN (facial)', 'LogReg probe (embeddings)', 'XGBoost Q-CHAT-10'],
    'AUC-ROC'    : [f'{facial_auc:.4f}',  f'{logreg_auc:.4f}',         f'{qchat_auc:.4f}'],
    'Recall'     : [f'{facial_m["recall"]:.4f}',  f'{logreg_m["recall"]:.4f}',  f'{qchat_m["recall"]:.4f}'],
    'Precision'  : [f'{facial_m["precision"]:.4f}', f'{logreg_m["precision"]:.4f}', f'{qchat_m["precision"]:.4f}'],
    'Specificity': [f'{facial_m["specificity"]:.4f}', f'{logreg_m["specificity"]:.4f}', f'{qchat_m["specificity"]:.4f}'],
    'F2-Score'   : [f'{facial_m["f2"]:.4f}',  f'{logreg_m["f2"]:.4f}',  f'{qchat_m["f2"]:.4f}'],
})

print('Standalone Baseline Metrics (threshold = 0.50):')
print(baseline_df.to_string(index=False))
print(f'\nLogreg probe AUC vs VGG-Face CNN: {logreg_auc:.4f} vs {facial_auc:.4f}')
print('(If LogReg ~= CNN: embeddings carry most discriminative information)')

---
## Section B — Alpha Sweep: Monte Carlo Simulation

For each alpha value in `np.arange(0.0, 1.05, 0.05)` (21 values):
- Run **500 Monte Carlo iterations**
- Each iteration: sample **150 ASD pairs + 150 non-ASD pairs** (stratified, with replacement)
  - For each pair: one facial sample + one Q-CHAT sample, **matched by class label**
  - `P_final = α × P_facial + (1−α) × P_qchat`
- Compute AUC-ROC, F2 (β=2), Recall, Precision, Specificity at threshold=0.50
- Report mean ± std across 500 iterations

In [ ]:
def run_alpha_sweep(df_facial, df_qchat, alpha_values,
                    n_iter=N_MC, n_asd=N_PAIRS_ASD, n_non=N_PAIRS_NON,
                    random_state=RANDOM_STATE):
    """
    Monte Carlo alpha sweep.
    Pairs are formed by matching class labels (ASD-ASD, non-non).
    Returns list of metric dicts, one per alpha value.
    """
    rng = np.random.RandomState(random_state)

    f_asd = df_facial[df_facial['Actual_Label'] == 1]['Model_Prob'].values
    f_non = df_facial[df_facial['Actual_Label'] == 0]['Model_Prob'].values
    q_asd = df_qchat[df_qchat['true_label']   == 1]['P_ASD'].values
    q_non = df_qchat[df_qchat['true_label']   == 0]['P_ASD'].values

    results = []
    for alpha in alpha_values:
        aucs, f2s, recalls, precisions, specificities = [], [], [], [], []

        for _ in range(n_iter):
            pf = np.concatenate([
                rng.choice(f_asd, n_asd, replace=True),
                rng.choice(f_non, n_non, replace=True)
            ])
            pq = np.concatenate([
                rng.choice(q_asd, n_asd, replace=True),
                rng.choice(q_non, n_non, replace=True)
            ])
            y  = np.array([1] * n_asd + [0] * n_non)

            p_final = alpha * pf + (1.0 - alpha) * pq
            y_pred  = (p_final >= 0.5).astype(int)

            try:
                aucs.append(roc_auc_score(y, p_final))
            except ValueError:
                aucs.append(np.nan)

            tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
            recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
            precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0.0)
            specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
            f2s.append(fbeta_score(y, y_pred, beta=2, zero_division=0))

        row = {
            'alpha'      : round(float(alpha), 2),
            'auc_mean'   : float(np.nanmean(aucs)),
            'auc_std'    : float(np.nanstd(aucs)),
            'f2_mean'    : float(np.mean(f2s)),
            'f2_std'     : float(np.std(f2s)),
            'recall_mean': float(np.mean(recalls)),
            'recall_std' : float(np.std(recalls)),
            'prec_mean'  : float(np.mean(precisions)),
            'prec_std'   : float(np.std(precisions)),
            'spec_mean'  : float(np.mean(specificities)),
            'spec_std'   : float(np.std(specificities)),
        }
        results.append(row)
        print(f'  alpha={alpha:.2f}  AUC={row["auc_mean"]:.4f}±{row["auc_std"]:.4f}'
              f'  F2={row["f2_mean"]:.4f}  Recall={row["recall_mean"]:.4f}')

    return results

In [ ]:
print(f'Running alpha sweep: {len(ALPHA_VALUES)} alphas x {N_MC} iterations x {N_PAIRS} pairs ...')
print()
sweep_results = run_alpha_sweep(df_img, df_qc, ALPHA_VALUES)
print('\nAlpha sweep complete.')

In [ ]:
# ─── Find optimal alpha ───────────────────────────────────────────────────────
sweep_df = pd.DataFrame(sweep_results)

idx_auc      = sweep_df['auc_mean'].idxmax()
idx_f2       = sweep_df['f2_mean'].idxmax()
alpha_opt_auc = sweep_df.loc[idx_auc, 'alpha']
alpha_opt_f2  = sweep_df.loc[idx_f2,  'alpha']
alpha_opt     = alpha_opt_auc   # primary optimal
opt_row       = sweep_df.loc[idx_auc]

best_standalone = max(facial_auc, qchat_auc)
auc_gain        = opt_row['auc_mean'] - best_standalone

print(f'Optimal alpha (AUC-based)  : {alpha_opt_auc:.2f}')
print(f'  Fused AUC                : {opt_row["auc_mean"]:.4f} +/- {opt_row["auc_std"]:.4f}')
print(f'  Fused F2                 : {opt_row["f2_mean"]:.4f} +/- {opt_row["f2_std"]:.4f}')
print(f'  Fused Recall             : {opt_row["recall_mean"]:.4f}')
print()
print(f'Optimal alpha (F2-based)   : {alpha_opt_f2:.2f}')
print()
print(f'Standalone AUC: facial={facial_auc:.4f}  qchat={qchat_auc:.4f}  '
      f'logreg_probe={logreg_auc:.4f}')
print(f'Best standalone AUC        : {best_standalone:.4f}')
print(f'Fusion AUC gain            : {auc_gain:+.4f}')

print()
print('Full alpha sweep table:')
disp = sweep_df[['alpha','auc_mean','auc_std','f2_mean','f2_std',
                  'recall_mean','prec_mean','spec_mean']].copy()
disp.columns = ['alpha','AUC_mu','AUC_sd','F2_mu','F2_sd','Recall_mu','Prec_mu','Spec_mu']
print(disp.round(4).to_string(index=False))

---
## Section C — Failure Mode Complementarity Analysis

On ASD-positive cases only:
1. **When facial misses** (FN) — what fraction does Q-CHAT catch?
2. **When Q-CHAT misses** (FN) — what fraction does facial catch?
3. **Both miss** — irreducible joint FN rate

In [ ]:
def complementarity_analysis(df_facial, df_qchat, alpha,
                               n_iter=N_MC, n_asd=N_PAIRS_ASD,
                               random_state=RANDOM_STATE):
    """Failure mode complementarity on ASD-positive cases."""
    rng = np.random.RandomState(random_state)

    f_asd = df_facial[df_facial['Actual_Label'] == 1]['Model_Prob'].values
    q_asd = df_qchat[df_qchat['true_label']   == 1]['P_ASD'].values

    qchat_rescues_facial = []
    facial_rescues_qchat = []
    both_fn_rates        = []

    for _ in range(n_iter):
        pf = rng.choice(f_asd, n_asd, replace=True)
        pq = rng.choice(q_asd, n_asd, replace=True)

        f_pred = (pf >= 0.5).astype(int)
        q_pred = (pq >= 0.5).astype(int)

        # Facial FN: facial says 0 on true ASD — does Q-CHAT say 1?
        f_fn_mask = (f_pred == 0)
        if f_fn_mask.sum() > 0:
            qchat_rescues_facial.append(q_pred[f_fn_mask].mean())
        else:
            qchat_rescues_facial.append(np.nan)

        # Q-CHAT FN: Q-CHAT says 0 on true ASD — does facial say 1?
        q_fn_mask = (q_pred == 0)
        if q_fn_mask.sum() > 0:
            facial_rescues_qchat.append(f_pred[q_fn_mask].mean())
        else:
            facial_rescues_qchat.append(np.nan)

        # Both miss
        both_fn_rates.append(((f_pred == 0) & (q_pred == 0)).mean())

    return {
        'facial_standalone_asd_recall' : float(np.mean(f_asd >= 0.5)),
        'qchat_standalone_asd_recall'  : float(np.mean(q_asd >= 0.5)),
        'qchat_rescues_facial_fn_mean' : float(np.nanmean(qchat_rescues_facial)),
        'qchat_rescues_facial_fn_std'  : float(np.nanstd(qchat_rescues_facial)),
        'facial_rescues_qchat_fn_mean' : float(np.nanmean(facial_rescues_qchat)),
        'facial_rescues_qchat_fn_std'  : float(np.nanstd(facial_rescues_qchat)),
        'both_fn_rate_mean'            : float(np.mean(both_fn_rates)),
        'both_fn_rate_std'             : float(np.std(both_fn_rates)),
    }


print(f'Running complementarity analysis at alpha={alpha_opt:.2f} ...')
comp_results = complementarity_analysis(df_img, df_qc, alpha_opt)

print()
print('Failure Mode Complementarity (ASD-positive cases only):')
print(f'  Facial standalone ASD recall       : {comp_results["facial_standalone_asd_recall"]:.1%}')
print(f'  Q-CHAT standalone ASD recall       : {comp_results["qchat_standalone_asd_recall"]:.1%}')
print(f'  When facial misses -> Q-CHAT catches: {comp_results["qchat_rescues_facial_fn_mean"]:.1%} '
      f'+/- {comp_results["qchat_rescues_facial_fn_std"]:.1%}')
print(f'  When Q-CHAT misses -> facial catches: {comp_results["facial_rescues_qchat_fn_mean"]:.1%} '
      f'+/- {comp_results["facial_rescues_qchat_fn_std"]:.1%}')
print(f'  Both miss (irreducible FN rate)    : {comp_results["both_fn_rate_mean"]:.1%} '
      f'+/- {comp_results["both_fn_rate_std"]:.1%}')

---
## Section D — Threshold Sweep at Optimal Alpha

Sweep thresholds τ ∈ [0.10, 0.90] to identify:
- **F2-optimal τ**: maximises F2-score (β=2, recall-biased)
- **Clinical τ**: highest τ that maintains Recall ≥ 80% (minimises false negatives in screening)

In [ ]:
THRESHOLDS = np.arange(0.10, 0.91, 0.05)

def threshold_sweep(df_facial, df_qchat, alpha,
                    thresholds=THRESHOLDS,
                    n_iter=N_MC, n_asd=N_PAIRS_ASD, n_non=N_PAIRS_NON,
                    random_state=RANDOM_STATE):
    """Threshold sweep at a fixed alpha via Monte Carlo."""
    rng = np.random.RandomState(random_state + 10)  # different seed from sweep

    f_asd = df_facial[df_facial['Actual_Label'] == 1]['Model_Prob'].values
    f_non = df_facial[df_facial['Actual_Label'] == 0]['Model_Prob'].values
    q_asd = df_qchat[df_qchat['true_label']   == 1]['P_ASD'].values
    q_non = df_qchat[df_qchat['true_label']   == 0]['P_ASD'].values

    store = {t: {'recall': [], 'precision': [], 'specificity': [],
                 'f1': [], 'f2': []}
             for t in thresholds}

    for _ in range(n_iter):
        pf = np.concatenate([
            rng.choice(f_asd, n_asd, replace=True),
            rng.choice(f_non, n_non, replace=True)
        ])
        pq = np.concatenate([
            rng.choice(q_asd, n_asd, replace=True),
            rng.choice(q_non, n_non, replace=True)
        ])
        y       = np.array([1] * n_asd + [0] * n_non)
        p_final = alpha * pf + (1.0 - alpha) * pq

        for t in thresholds:
            y_pred = (p_final >= t).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
            store[t]['recall'].append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
            store[t]['precision'].append(tp / (tp + fp) if (tp + fp) > 0 else 0.0)
            store[t]['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
            store[t]['f1'].append(fbeta_score(y, y_pred, beta=1, zero_division=0))
            store[t]['f2'].append(fbeta_score(y, y_pred, beta=2, zero_division=0))

    rows = []
    for t in thresholds:
        rows.append({
            'threshold'      : round(float(t), 2),
            'recall_mean'    : float(np.mean(store[t]['recall'])),
            'recall_std'     : float(np.std(store[t]['recall'])),
            'precision_mean' : float(np.mean(store[t]['precision'])),
            'precision_std'  : float(np.std(store[t]['precision'])),
            'specificity_mean': float(np.mean(store[t]['specificity'])),
            'specificity_std': float(np.std(store[t]['specificity'])),
            'f1_mean'        : float(np.mean(store[t]['f1'])),
            'f1_std'         : float(np.std(store[t]['f1'])),
            'f2_mean'        : float(np.mean(store[t]['f2'])),
            'f2_std'         : float(np.std(store[t]['f2'])),
        })
    return pd.DataFrame(rows)


print(f'Running threshold sweep at alpha={alpha_opt:.2f} ({len(THRESHOLDS)} thresholds x {N_MC} iterations)...')
thresh_df = threshold_sweep(df_img, df_qc, alpha_opt)

# F2-optimal threshold
idx_f2_opt   = thresh_df['f2_mean'].idxmax()
opt_thresh_row = thresh_df.loc[idx_f2_opt]
opt_thresh     = opt_thresh_row['threshold']

# Clinical threshold: highest tau with mean recall >= 0.80
clinical_mask = thresh_df['recall_mean'] >= 0.80
if clinical_mask.any():
    clinical_thresh_row = thresh_df[clinical_mask].iloc[-1]
else:
    clinical_thresh_row = thresh_df.loc[thresh_df['recall_mean'].idxmax()]
clinical_thresh = clinical_thresh_row['threshold']

print()
print(f'F2-optimal threshold (tau)       : {opt_thresh:.2f}')
print(f'  Recall       = {opt_thresh_row["recall_mean"]:.1%} +/- {opt_thresh_row["recall_std"]:.1%}')
print(f'  Precision    = {opt_thresh_row["precision_mean"]:.1%} +/- {opt_thresh_row["precision_std"]:.1%}')
print(f'  Specificity  = {opt_thresh_row["specificity_mean"]:.1%}')
print(f'  F2-Score     = {opt_thresh_row["f2_mean"]:.4f}')
print()
print(f'Clinical threshold (recall >= 80%): {clinical_thresh:.2f}')
print(f'  Recall       = {clinical_thresh_row["recall_mean"]:.1%}')
print(f'  Specificity  = {clinical_thresh_row["specificity_mean"]:.1%}')
print()
print('Full threshold sweep:')
disp = thresh_df[['threshold','recall_mean','precision_mean',
                   'specificity_mean','f1_mean','f2_mean']].copy()
disp.columns = ['tau','Recall_mu','Prec_mu','Spec_mu','F1_mu','F2_mu']
print(disp.round(4).to_string(index=False))

---
## Section E — Visualizations

Five plots saved to `sector3/plots/`:
1. `alpha_sweep_auc.png` — AUC vs alpha with error bands
2. `alpha_sweep_f2.png` — F2 vs alpha with error bands
3. `roc_comparison.png` — ROC curves (facial, Q-CHAT, fused) from pooled MC
4. `complementarity_analysis.png` — bar chart of failure mode rescue rates
5. `threshold_analysis_fusion.png` — threshold vs Recall/Precision/F2

In [ ]:
# ── Plot 1: AUC vs Alpha ──────────────────────────────────────────────────────
alphas   = sweep_df['alpha'].values
auc_mean = sweep_df['auc_mean'].values
auc_std  = sweep_df['auc_std'].values

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(alphas,
                auc_mean - auc_std, auc_mean + auc_std,
                alpha=0.20, color='steelblue', label='±1σ band')
ax.plot(alphas, auc_mean, 'o-', color='steelblue',
        linewidth=2, markersize=5, label='Fused AUC')
ax.axhline(facial_auc, ls='--', color='tomato',   linewidth=1.5,
           label=f'Facial only  ({facial_auc:.3f})')
ax.axhline(qchat_auc,  ls='--', color='seagreen', linewidth=1.5,
           label=f'Q-CHAT only  ({qchat_auc:.3f})')
ax.axvline(alpha_opt,  ls=':',  color='black',    linewidth=1.5,
           label=f'Optimal alpha={alpha_opt:.2f}')
ax.set_xlabel('Alpha (facial stream weight)')
ax.set_ylabel('AUC-ROC')
ax.set_title(f'Fusion AUC-ROC vs Alpha\n({N_MC} MC iterations x {N_PAIRS} stratified pairs each)')
ax.legend(loc='best')
ax.set_xlim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'alpha_sweep_auc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR / "alpha_sweep_auc.png"}')

In [ ]:
# ── Plot 2: F2 vs Alpha ───────────────────────────────────────────────────────
f2_mean = sweep_df['f2_mean'].values
f2_std  = sweep_df['f2_std'].values

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(alphas,
                f2_mean - f2_std, f2_mean + f2_std,
                alpha=0.20, color='darkorange', label='±1σ band')
ax.plot(alphas, f2_mean, 'o-', color='darkorange',
        linewidth=2, markersize=5, label='Fused F2 (beta=2)')
ax.axvline(alpha_opt_f2, ls=':', color='black', linewidth=1.5,
           label=f'F2-optimal alpha={alpha_opt_f2:.2f}')
ax.axvline(alpha_opt_auc, ls='--', color='steelblue', linewidth=1.2,
           label=f'AUC-optimal alpha={alpha_opt_auc:.2f}')
ax.set_xlabel('Alpha (facial stream weight)')
ax.set_ylabel('F2-Score (beta=2, recall-biased)')
ax.set_title(f'Fusion F2-Score vs Alpha\n({N_MC} MC iterations x {N_PAIRS} stratified pairs each)')
ax.legend(loc='best')
ax.set_xlim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'alpha_sweep_f2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR / "alpha_sweep_f2.png"}')

In [ ]:
# ── Plot 3: ROC Comparison (pooled across 500 MC iterations) ──────────────────
rng_roc = np.random.RandomState(RANDOM_STATE + 99)

f_asd_all = df_img[df_img['Actual_Label'] == 1]['Model_Prob'].values
f_non_all = df_img[df_img['Actual_Label'] == 0]['Model_Prob'].values
q_asd_all = df_qc[df_qc['true_label']   == 1]['P_ASD'].values
q_non_all = df_qc[df_qc['true_label']   == 0]['P_ASD'].values

pool_pf, pool_pq, pool_pfused, pool_y = [], [], [], []
for _ in range(N_MC):
    pf = np.concatenate([
        rng_roc.choice(f_asd_all, N_PAIRS_ASD, replace=True),
        rng_roc.choice(f_non_all, N_PAIRS_NON, replace=True)
    ])
    pq = np.concatenate([
        rng_roc.choice(q_asd_all, N_PAIRS_ASD, replace=True),
        rng_roc.choice(q_non_all, N_PAIRS_NON, replace=True)
    ])
    y = np.array([1] * N_PAIRS_ASD + [0] * N_PAIRS_NON)
    pool_pf.append(pf)
    pool_pq.append(pq)
    pool_pfused.append(alpha_opt * pf + (1.0 - alpha_opt) * pq)
    pool_y.append(y)

pool_pf    = np.concatenate(pool_pf)
pool_pq    = np.concatenate(pool_pq)
pool_pfused = np.concatenate(pool_pfused)
pool_y     = np.concatenate(pool_y)

fpr_f, tpr_f, _ = roc_curve(pool_y, pool_pf)
fpr_q, tpr_q, _ = roc_curve(pool_y, pool_pq)
fpr_o, tpr_o, _ = roc_curve(pool_y, pool_pfused)

auc_f_pool = roc_auc_score(pool_y, pool_pf)
auc_q_pool = roc_auc_score(pool_y, pool_pq)
auc_o_pool = roc_auc_score(pool_y, pool_pfused)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_f, tpr_f, color='tomato',    lw=2,
        label=f'Facial only   AUC={auc_f_pool:.3f}')
ax.plot(fpr_q, tpr_q, color='seagreen',  lw=2,
        label=f'Q-CHAT only   AUC={auc_q_pool:.3f}')
ax.plot(fpr_o, tpr_o, color='steelblue', lw=2.5,
        label=f'Fused (alpha={alpha_opt:.2f})   AUC={auc_o_pool:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title(f'ROC Comparison — Pooled MC ({N_MC} iterations x {N_PAIRS} pairs)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR / "roc_comparison.png"}')

In [ ]:
# ── Plot 4: Failure Mode Complementarity ──────────────────────────────────────
bar_labels = [
    'Facial ASD\nRecall',
    'Q-CHAT ASD\nRecall',
    'Q-CHAT rescues\nFacial FN',
    'Facial rescues\nQ-CHAT FN',
    'Irreducible\nBoth-FN Rate',
]
bar_values = [
    comp_results['facial_standalone_asd_recall'],
    comp_results['qchat_standalone_asd_recall'],
    comp_results['qchat_rescues_facial_fn_mean'],
    comp_results['facial_rescues_qchat_fn_mean'],
    comp_results['both_fn_rate_mean'],
]
bar_errors = [
    0, 0,
    comp_results['qchat_rescues_facial_fn_std'],
    comp_results['facial_rescues_qchat_fn_std'],
    comp_results['both_fn_rate_std'],
]
bar_colors = ['tomato', 'seagreen', '#4C9BE8', '#E88D4C', '#888888']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(bar_labels, bar_values, yerr=bar_errors, capsize=6,
              color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + 0.025,
            f'{val:.1%}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Rate')
ax.set_title(f'Failure Mode Complementarity (alpha={alpha_opt:.2f}, ASD cases only)\n'
             f'Error bars = ±1σ across {N_MC} MC iterations')
ax.set_ylim([0, 1.18])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'complementarity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR / "complementarity_analysis.png"}')

In [ ]:
# ── Plot 5: Threshold Analysis at Optimal Alpha ────────────────────────────────
t_vals = thresh_df['threshold'].values

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_vals, thresh_df['recall_mean'],     'o-', color='tomato',    lw=2,   label='Recall')
ax.plot(t_vals, thresh_df['precision_mean'],  's-', color='steelblue', lw=2,   label='Precision')
ax.plot(t_vals, thresh_df['specificity_mean'],'D-', color='seagreen',  lw=2,   label='Specificity')
ax.plot(t_vals, thresh_df['f2_mean'],         '^-', color='darkorange',lw=2.5, label='F2-Score (beta=2)')

ax.fill_between(t_vals,
                thresh_df['recall_mean'] - thresh_df['recall_std'],
                thresh_df['recall_mean'] + thresh_df['recall_std'],
                alpha=0.10, color='tomato')

ax.axvline(opt_thresh,      ls='--', color='darkorange', lw=1.5,
           label=f'F2-optimal tau={opt_thresh:.2f}')
ax.axvline(clinical_thresh, ls=':',  color='black',      lw=1.5,
           label=f'Clinical tau={clinical_thresh:.2f} (recall>=80%)')
ax.set_xlabel('Decision Threshold (tau)')
ax.set_ylabel('Metric Value')
ax.set_title(f'Threshold Analysis at alpha={alpha_opt:.2f}\n'
             f'({N_MC} MC iterations x {N_PAIRS} stratified pairs)')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
ax.set_xlim([0.05, 0.95])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'threshold_analysis_fusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR / "threshold_analysis_fusion.png"}')

---
## Section F — Clinical Interpretation

Summary of simulation findings with clinical significance.

In [ ]:
print('=' * 72)
print('  SECTION F — CLINICAL INTERPRETATION')
print('=' * 72)

print()
print('FUSION CONFIGURATION')
print(f'  Optimal alpha (AUC)     : {alpha_opt:.2f}')
print(f'  Facial weight           : {alpha_opt:.0%}')
print(f'  Q-CHAT weight           : {1.0 - alpha_opt:.0%}')
print(f'  Fused AUC               : {opt_row["auc_mean"]:.4f} +/- {opt_row["auc_std"]:.4f}')
print(f'  Fused F2                : {opt_row["f2_mean"]:.4f} +/- {opt_row["f2_std"]:.4f}')
print(f'  Fused Recall @ tau=0.5  : {opt_row["recall_mean"]:.1%}')

print()
print('STANDALONE vs FUSION (AUC):')
print(f'  Facial only             : {facial_auc:.4f}')
print(f'  Q-CHAT only             : {qchat_auc:.4f}')
print(f'  LogReg probe            : {logreg_auc:.4f}')
print(f'  Fused (simulated)       : {opt_row["auc_mean"]:.4f}')
print(f'  AUC gain over best      : {auc_gain:+.4f}')

print()
print('COMPLEMENTARITY (ASD CASES):')
print(f'  Facial ASD recall       : {comp_results["facial_standalone_asd_recall"]:.1%}')
print(f'  Q-CHAT ASD recall       : {comp_results["qchat_standalone_asd_recall"]:.1%}')
print(f'  Q-CHAT rescues facial FN: {comp_results["qchat_rescues_facial_fn_mean"]:.1%}')
print(f'  Facial rescues Q-CHAT FN: {comp_results["facial_rescues_qchat_fn_mean"]:.1%}')
print(f'  Irreducible both-FN rate: {comp_results["both_fn_rate_mean"]:.1%}')

print()
print('THRESHOLD RECOMMENDATIONS:')
print(f'  F2-optimal (tau)        : {opt_thresh:.2f}')
print(f'    Recall    = {opt_thresh_row["recall_mean"]:.1%}')
print(f'    Precision = {opt_thresh_row["precision_mean"]:.1%}')
print(f'    F2-Score  = {opt_thresh_row["f2_mean"]:.4f}')
print(f'  Clinical (recall>=80%): {clinical_thresh:.2f}')
print(f'    Recall    = {clinical_thresh_row["recall_mean"]:.1%}')
print(f'    Specificity = {clinical_thresh_row["specificity_mean"]:.1%}')

print()
print('INTERPRETATION:')
if auc_gain > 0.01:
    print('  -> Fusion provides meaningful AUC improvement over either modality alone.')
elif auc_gain > 0:
    print('  -> Fusion provides marginal AUC improvement; value lies in complementarity.')
else:
    print('  -> Fusion does not improve AUC; best standalone model dominates.')
print(f'  -> Complementarity is the key benefit: when one stream misses, the other')
print(f'     partially compensates, reducing the irreducible FN rate to')
print(f'     ~{comp_results["both_fn_rate_mean"]:.1%} of ASD cases.')
print()
print('CAVEAT:')
print(f'  Results based on {N_MC} MC iterations x {N_PAIRS} stratified pairs.')
print('  Datasets have no patient overlap. Simulation models expected')
print('  population-level behaviour only. Prospective paired data collection')
print('  is required for definitive validation.')
print()
print('=' * 72)

---
## Section G — Save Results

In [ ]:
summary = {
    'simulation_config': {
        'n_mc_iterations' : N_MC,
        'n_pairs_per_iter': N_PAIRS,
        'n_pairs_asd'     : N_PAIRS_ASD,
        'n_pairs_non_asd' : N_PAIRS_NON,
        'alpha_values'    : [round(float(a), 2) for a in ALPHA_VALUES],
        'random_state'    : RANDOM_STATE,
        'threshold_default': 0.50,
    },
    'datasets': {
        'facial_n'     : int(len(df_img)),
        'facial_asd_n' : int((df_img['Actual_Label'] == 1).sum()),
        'facial_non_n' : int((df_img['Actual_Label'] == 0).sum()),
        'qchat_n'      : int(len(df_qc)),
        'qchat_asd_n'  : int((df_qc['true_label'] == 1).sum()),
        'qchat_non_n'  : int((df_qc['true_label'] == 0).sum()),
        'note'         : 'No patient overlap between facial and Q-CHAT datasets',
    },
    'standalone_baselines': {
        'facial_auc'       : round(facial_auc, 4),
        'qchat_auc'        : round(qchat_auc, 4),
        'logreg_probe_auc' : round(logreg_auc, 4),
        'facial_recall'    : round(facial_m['recall'], 4),
        'facial_precision' : round(facial_m['precision'], 4),
        'facial_f2'        : round(facial_m['f2'], 4),
        'qchat_recall'     : round(qchat_m['recall'], 4),
        'qchat_precision'  : round(qchat_m['precision'], 4),
        'qchat_f2'         : round(qchat_m['f2'], 4),
    },
    'alpha_sweep': sweep_results,
    'optimal_fusion': {
        'alpha_opt_auc'                  : float(alpha_opt_auc),
        'alpha_opt_f2'                   : float(alpha_opt_f2),
        'fused_auc_at_opt'               : round(float(opt_row['auc_mean']), 4),
        'fused_auc_std'                  : round(float(opt_row['auc_std']), 4),
        'fused_f2_at_opt'                : round(float(opt_row['f2_mean']), 4),
        'fused_f2_std'                   : round(float(opt_row['f2_std']), 4),
        'fused_recall'                   : round(float(opt_row['recall_mean']), 4),
        'best_standalone_auc'            : round(best_standalone, 4),
        'auc_gain_over_best_standalone'  : round(auc_gain, 4),
    },
    'complementarity': comp_results,
    'threshold_analysis': {
        'alpha_used'          : float(alpha_opt),
        'f2_optimal_threshold': float(opt_thresh),
        'f2_optimal_recall'   : round(float(opt_thresh_row['recall_mean']), 4),
        'f2_optimal_precision': round(float(opt_thresh_row['precision_mean']), 4),
        'f2_optimal_f2'       : round(float(opt_thresh_row['f2_mean']), 4),
        'clinical_threshold'  : float(clinical_thresh),
        'clinical_recall'     : round(float(clinical_thresh_row['recall_mean']), 4),
        'clinical_specificity': round(float(clinical_thresh_row['specificity_mean']), 4),
        'full_sweep'          : thresh_df.round(4).to_dict('records'),
    },
    'plots_saved': [
        'plots/alpha_sweep_auc.png',
        'plots/alpha_sweep_f2.png',
        'plots/roc_comparison.png',
        'plots/complementarity_analysis.png',
        'plots/threshold_analysis_fusion.png',
    ],
}

out_path = RESULTS_DIR / 'fusion_simulation_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved: {out_path}')
print()
print('Fusion simulation complete.')
print(f'  Optimal alpha         : {alpha_opt:.2f}')
print(f'  Best fused AUC        : {opt_row["auc_mean"]:.4f} +/- {opt_row["auc_std"]:.4f}')
print(f'  AUC gain over solo    : {auc_gain:+.4f}')
print(f'  F2-optimal threshold  : {opt_thresh:.2f}')
print(f'  Clinical threshold    : {clinical_thresh:.2f}')
print(f'  Plots saved to        : {PLOTS_DIR}')
print(f'  Results saved to      : {RESULTS_DIR}')